# Task 1 — Step 1: Standard Direct Preference Optimization (DPO)

### Mathematical Formulation
Direct Preference Optimization models policy preferences directly without training an intermediate reward model or sampling on-policy trajectories:
$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y^+, y^-)}\left[\log \sigma\left(\beta \left[\log\frac{\pi_\theta(y^+|x)}{\pi_{\text{ref}}(y^+|x)} - \log\frac{\pi_\theta(y^-|x)}{\pi_{\text{ref}}(y^-|x)}\right]\right)\right]$$

where:
- $x$ is the prompt, $y^+$ is the preferred response, and $y^-$ is the rejected response.
- $\pi_{\text{ref}}$ is the frozen reference policy (`Qwen2.5-1.5B-Instruct`).
- $\beta$ controls the regularization strength relative to the reference policy.

### Required Evidence (from Course Manual):
1. **Training trajectory**: DPO loss, batch preference accuracy, implicit reward margin, and sequence KL estimate over 1 epoch (1500 pairs).
2. **Evaluation metrics**: Held-out preference accuracy on 300 test pairs, mean KL drift from reference, scalar reward-model score on 100 generated responses, and response-length statistics (mean and std).
3. **Qualitative completions**: Top 3 highest-reward and bottom 3 lowest-reward generated responses.


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Execution Settings
Set `FORCE_RUN = True` to re-run training and evaluation, or `False` to inspect saved artifacts.


In [ ]:
FORCE_RUN = False
CONFIG = "configs/dpo.yaml"
OUTPUT_DIR = "outputs/task1_dpo/standard"
RESULTS_DIR = "results/task1_dpo"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs("report/figures", exist_ok=True)


## 2. Train Standard DPO (1 Epoch)


In [ ]:
if FORCE_RUN or not os.path.exists(OUTPUT_DIR):
    !python -m task1_dpo.train --config configs/dpo.yaml --run-name standard
else:
    print(f"[OK] Trained adapter found at {OUTPUT_DIR}")


## 3. Evaluate Standard DPO on Held-Out UltraFeedback Pairs


In [ ]:
eval_file = os.path.join(RESULTS_DIR, "dpo_eval_standard.json")
if FORCE_RUN or not os.path.exists(eval_file):
    !python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard
else:
    print(f"[OK] Evaluation results found at {eval_file}")


## 4. Summary Table of Standard DPO Metrics


In [ ]:
if os.path.exists(eval_file):
    with open(eval_file, encoding='utf-8') as f:
        eval_data = json.load(f)
    
    summary_df = pd.DataFrame([{
        "Metric": "Held-out Preference Accuracy",
        "Value": f"{eval_data.get('held_out_preference_accuracy', 0)*100:.2f}%",
        "Target/Baseline": "Random guess: 50%"
    }, {
        "Metric": "Policy Drift (Mean KL from Ref)",
        "Value": f"{eval_data.get('mean_kl_from_reference', 0):.4f}",
        "Target/Baseline": "Reference = 0.0"
    }, {
        "Metric": "Mean Reward-Model Score",
        "Value": f"{eval_data.get('mean_reward', 0):.3f} ± {eval_data.get('std_reward', 0):.3f}",
        "Target/Baseline": "SFT baseline ~ 0.8"
    }, {
        "Metric": "Mean Response Length (words)",
        "Value": f"{eval_data.get('mean_response_length_words', 0):.1f}",
        "Target/Baseline": "Word limit target: 40"
    }, {
        "Metric": "Word-Limit Compliance Rate",
        "Value": f"{eval_data.get('word_limit_compliance_rate', 0)*100:.1f}%",
        "Target/Baseline": "Higher is better"
    }])
    display(summary_df)


## 5. Visualizing DPO Training Dynamics


In [ ]:
log_file = os.path.join(RESULTS_DIR, "dpo_train_standard.jsonl")
if os.path.exists(log_file):
    records = [json.loads(line) for line in open(log_file, encoding='utf-8')]
    df = pd.DataFrame(records)
    
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    
    # 1. Loss
    axes[0].plot(df['step'], df['loss'], color='#2563EB', lw=2)
    axes[0].set_title('DPO Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)
    
    # 2. Preference Accuracy
    axes[1].plot(df['step'], df['pref_acc']*100, color='#10B981', lw=2)
    axes[1].axhline(50, color='gray', linestyle='--', alpha=0.7)
    axes[1].set_title('Batch Preference Accuracy (%)')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].grid(True, alpha=0.3)
    
    # 3. Logit Margin
    axes[2].plot(df['step'], df['logit_mean'], color='#8B5CF6', lw=2)
    axes[2].axhline(0, color='gray', linestyle='--', alpha=0.7)
    axes[2].set_title('Mean Implicit Reward Margin')
    axes[2].set_xlabel('Step')
    axes[2].set_ylabel('Margin')
    axes[2].grid(True, alpha=0.3)
    
    # 4. KL Drift
    axes[3].plot(df['step'], df['kl'], color='#F59E0B', lw=2)
    axes[3].set_title('Sequence KL Drift')
    axes[3].set_xlabel('Step')
    axes[3].set_ylabel('KL')
    axes[3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


## 6. Publication Figures Inspection


In [ ]:
fig_paths = [
    "report/figures/task1_dpo_training_curves.png",
    "report/figures/task1_dpo_eval_standard.png"
]
for p in fig_paths:
    if os.path.exists(p):
        print(f"Displaying publication figure: {p}")
        display(Image(filename=p, width=650))


## 7. Qualitative Samples Inspection (Reward vs. Quality)
The manual requires showing examples where reward model scores either align or conflict with actual response quality.


In [ ]:
if os.path.exists(eval_file):
    per_ex = eval_data.get("per_example", [])
    if per_ex:
        sorted_ex = sorted(per_ex, key=lambda x: x.get("reward", 0), reverse=True)
        print("=" * 80)
        print("TOP 2 HIGHEST REWARD RESPONSES:")
        print("=" * 80)
        for i, item in enumerate(sorted_ex[:2], 1):
            print(f"\n--- Top #{i} | Reward Score = {item.get('reward', 0):.3f} ---")
            print(f"PROMPT: {item.get('prompt', '')[:120]}...")
            print(f"RESPONSE: {item.get('response', '')[:250]}...")
            
        print("\n" + "=" * 80)
        print("BOTTOM 2 LOWEST REWARD RESPONSES:")
        print("=" * 80)
        for i, item in enumerate(sorted_ex[-2:], 1):
            print(f"\n--- Bottom #{i} | Reward Score = {item.get('reward', 0):.3f} ---")
            print(f"PROMPT: {item.get('prompt', '')[:120]}...")
            print(f"RESPONSE: {item.get('response', '')[:250]}...")


## 8. Research Question 1 Analysis & Takeaways
- **Did standard DPO improve held-out preference fitting?** Compare against the 50% random baseline.
- **Is stronger preference fitting accompanied by useful behavior or disproportionate policy drift?** Analyze KL drift trajectory.
